# Project 52 — Local Spreadsheet Analyst Agent

**Answer questions over CSV/XLSX data and generate business insights — fully local.**

| Component | Choice |
|-----------|--------|
| LLM runtime | Ollama (qwen3:8b) |
| Framework | LangChain |
| Data tools | pandas |
| Interface | Jupyter notebook |

## 1 · What You Will Learn

1. How to let an LLM **generate and execute** pandas code from natural-language questions
2. How to **safely sandbox** LLM-generated code with output capture
3. How to build a **data analyst copilot** that explains results in plain English
4. How to generate an **automated insights report** from tabular data
5. How to evaluate whether LLM-generated analysis is **correct**

## 2 · Why This Project Matters

Business users frequently need answers from spreadsheets but lack SQL or pandas skills.
A local spreadsheet analyst agent bridges this gap — users ask questions in plain
English and get data-backed answers. Running locally means sensitive financial data
never leaves your machine.

## 3 · Environment Setup

**Prerequisites:**
- Ollama running with `qwen3:8b`
- Python packages below

In [ ]:
# !pip install -q langchain langchain-ollama pandas openpyxl

## 4 · Imports and Configuration

In [ ]:
import pandas as pd
import io
import contextlib
import json
import os
from pathlib import Path

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL = "qwen3:8b"
llm = ChatOllama(model=MODEL, temperature=0.1)
print(f"LLM ready: {MODEL}")

## 5 · Create Sample Dataset

We create a realistic sales dataset. In production, you would load a user-uploaded
CSV/XLSX file instead.

In [ ]:
import numpy as np
np.random.seed(42)

months = pd.date_range('2024-01-01', periods=24, freq='MS')
regions = ['North', 'South', 'East', 'West']
products = ['Widget A', 'Widget B', 'Widget C', 'Premium D']

rows = []
for month in months:
    for region in regions:
        for product in products:
            base_rev = np.random.randint(5000, 25000)
            units = np.random.randint(50, 500)
            returns = np.random.randint(0, int(units * 0.08))
            rows.append({
                'date': month,
                'region': region,
                'product': product,
                'revenue': base_rev,
                'units_sold': units,
                'returns': returns,
                'cost': int(base_rev * np.random.uniform(0.4, 0.7)),
            })

df = pd.DataFrame(rows)
df['profit'] = df['revenue'] - df['cost']
df['return_rate'] = (df['returns'] / df['units_sold'] * 100).round(2)

Path('sample_data').mkdir(exist_ok=True)
df.to_csv('sample_data/sales_data.csv', index=False)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(8)

## 6 · Build the NL-to-Pandas Analyst

The core idea: the user asks a natural-language question, the LLM generates pandas
code, we execute it in a sandboxed environment, and return both the code and the result.

In [ ]:
def analyze_data(question: str, data: pd.DataFrame, verbose: bool = True) -> dict:
    """Generate and execute pandas code to answer a question about the data."""
    code_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a data analyst. Given a pandas DataFrame `df` with these columns:\n"
         "{columns}\n\n"
         "Data types:\n{dtypes}\n\n"
         "Sample rows:\n{sample}\n\n"
         "Write Python code using pandas to answer the question.\n"
         "Rules:\n"
         "- Use the variable `df` (already defined)\n"
         "- Use print() to show results\n"
         "- Return ONLY Python code, no explanations\n"
         "- Do not import pandas (already imported as pd)\n"
         "- Do not read any files (df is already loaded)"),
        ("human", "{question}"),
    ])

    code_chain = code_prompt | llm | StrOutputParser()
    generated = code_chain.invoke({
        "columns": str(list(data.columns)),
        "dtypes": data.dtypes.to_string(),
        "sample": data.head(3).to_string(),
        "question": question,
    })

    # Clean code (remove markdown fences)
    lines = []
    for line in generated.split("\n"):
        stripped = line.strip()
        if stripped and not stripped.startswith("```"):
            lines.append(line)  # preserve indentation
    clean_code = "\n".join(lines)

    # Execute safely
    output = io.StringIO()
    try:
        with contextlib.redirect_stdout(output):
            exec(clean_code, {"df": data, "pd": pd, "np": np, "print": print})
        result = output.getvalue().strip()
    except Exception as e:
        result = f"ERROR: {e}"

    if verbose:
        print(f"\nQ: {question}")
        print(f"Code:\n{clean_code}")
        print(f"Result:\n{result}")

    return {"question": question, "code": clean_code, "result": result}


print("Analyst function ready.")

## 7 · Ask Questions About the Data

Let's test the analyst with progressively harder questions.

In [ ]:
q1 = analyze_data("What is the total revenue by region?", df)

In [ ]:
q2 = analyze_data("Which product has the highest average profit margin?", df)

In [ ]:
q3 = analyze_data("Show the top 3 months by total revenue", df)

In [ ]:
q4 = analyze_data("What is the return rate trend over time? Show monthly averages.", df)

In [ ]:
q5 = analyze_data("Compare revenue per unit sold across regions and products", df)

## 8 · Generate an Insights Report

Beyond answering individual questions, a good analyst agent should be able to
produce a **comprehensive summary** of the data.

In [ ]:
# Build a data summary for the LLM
summary_stats = {
    "total_revenue": f"${df["revenue"].sum():,}",
    "total_profit": f"${df["profit"].sum():,}",
    "avg_monthly_revenue": f"${df.groupby("date")["revenue"].sum().mean():,.0f}",
    "revenue_by_region": df.groupby("region")["revenue"].sum().to_dict(),
    "revenue_by_product": df.groupby("product")["revenue"].sum().to_dict(),
    "avg_return_rate": f"{df["return_rate"].mean():.2f}%",
    "best_month": df.groupby("date")["revenue"].sum().idxmax().strftime("%B %Y"),
    "worst_month": df.groupby("date")["revenue"].sum().idxmin().strftime("%B %Y"),
}

insight_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a business analyst. Given sales data statistics, produce a structured\n"
     "report with:\n"
     "1. Executive Summary (3 sentences)\n"
     "2. Key Findings (5 bullet points)\n"
     "3. Risks and Concerns (3 bullet points)\n"
     "4. Recommendations (3 actionable items)\n"
     "Be specific and reference the numbers."),
    ("human", "Data Statistics:\n{stats}"),
])

insight_chain = insight_prompt | llm | StrOutputParser()
report = insight_chain.invoke({"stats": json.dumps(summary_stats, indent=2)})

print("=" * 60)
print("AUTOMATED BUSINESS INSIGHTS REPORT")
print("=" * 60)
print(report)

## 9 · Evaluate Analysis Quality

Let's verify the LLM-generated code actually produced correct results by comparing
against ground-truth pandas operations.

In [ ]:
# Ground truth checks
checks = [
    ("Total revenue", df["revenue"].sum(), q1["result"]),
    ("Best month present in Q3", df.groupby("date")["revenue"].sum().idxmax().strftime("%Y"), q3["result"]),
]

print("VALIDATION")
print("-" * 40)
for name, expected, result in checks:
    found = str(expected) in str(result)
    status = "PASS" if found else "CHECK"
    print(f"  {name}: {status} (expected {expected} in output)")

## 10 · Save Results

In [ ]:
results = {
    "questions": [q1, q2, q3, q4, q5],
    "report": report,
    "data_shape": list(df.shape),
}
with open("analysis_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print("Results saved to analysis_results.json")

## 11 · Limitations

- **Code execution** is sandboxed but not fully isolated — production systems need Docker/subprocess
- **Small models** can generate incorrect pandas syntax on complex queries
- **No retry logic** — if the first generated code fails, we just report the error
- **Column type inference** depends on the sample rows shown to the LLM
- Large datasets need sampling before showing to the LLM context window

## 12 · How to Improve

1. **Add retry logic**: if code execution fails, feed the error back to the LLM for correction
2. **Add visualization**: generate matplotlib plots alongside text answers
3. **Support XLSX**: add openpyxl-based loading for Excel files
4. **Add schema caching**: cache DataFrame schema to reduce prompt tokens
5. **Multi-DataFrame queries**: join across multiple spreadsheets

## 13 · Mini Challenge

1. Add a question that requires a `.groupby()` with multiple aggregation functions
2. Add a retry mechanism: if `analyze_data` returns an ERROR, re-prompt the LLM with the error
3. Create a visualization tool that generates a bar chart from the analysis

## 14 · Key Takeaways

| Concept | What You Practiced |
|---------|-------------------|
| NL-to-code | LLM generates executable pandas from English questions |
| Safe execution | Sandboxed `exec()` with captured stdout |
| Automated reporting | LLM produces structured business insights |
| Validation | Compare LLM results against ground truth |
| Local-first | No data leaves your machine |